### Sections
| | |
|---|---|
| **A** | Setup & load artefacts |
| **B** | Rebuild RAG pipeline with optimised prompt |
| **C** | Run 5 business queries |
| **D** | Response quality evaluation |
| **E** | Results dashboard — PPT-ready charts |
| **F** | Export report & artefact checklist |


In [ ]:
import os, sys, json, time, getpass, warnings
from dotenv import load_dotenv

from pathlib import Path
from typing  import List, Dict, Any, Optional

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sentence_transformers import SentenceTransformer
import torch, chromadb

from langchain_core.documents      import Document
from langchain_core.prompts        import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables      import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_groq                import ChatGroq

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR

In [ ]:
# ── Path configuration ──────

OUTPUT_DIR = DATA_DIR / 'outputs'
MODEL_DIR  = DATA_DIR / 'models'

# Load config from 03_01_rag_pipeline_core
RAG_CONFIG_FILE = OUTPUT_DIR / 'rag_config.json'
if not RAG_CONFIG_FILE.exists():
    raise FileNotFoundError('rag_config.json not found. Run 03_01_rag_pipeline_core first.')
with open(RAG_CONFIG_FILE) as f:
    cfg = json.load(f)

# Load optimised prompt from 03_02_prompt_engineering
PE_FILE = OUTPUT_DIR / 'prompt_engineering_output.json'
if not PE_FILE.exists():
    raise FileNotFoundError('prompt_engineering_output.json not found. Run 03_02_prompt_engineering first.')
with open(PE_FILE) as f:
    pe = json.load(f)

GROQ_MODEL_NAME  = cfg['groq_model_name']
COLLECTION_NAME  = cfg['collection_name']
FINE_TUNED_MODEL = cfg['embedding_model']
N_RESULTS        = cfg['n_results_default']
OPTIMISED_SYSTEM = pe['optimised_system']
FT_EMBEDDINGS    = OUTPUT_DIR / 'ft_embeddings.npy'
CHUNK_METADATA   = OUTPUT_DIR / 'chunk_metadata.json'

print(f' Groq model    : {GROQ_MODEL_NAME}')
print(f' Best strategy : {pe["best_strategy"]}')
print(f' Corpus size   : {cfg["total_chunks"]} chunks')

### Section B — Rebuild RAG Pipeline with Optimised Prompt

In [ ]:
embedding_model = SentenceTransformer(FINE_TUNED_MODEL, device=DEVICE)
ft_embeddings   = np.load(FT_EMBEDDINGS)

with open(CHUNK_METADATA) as f:
    metadata_list = json.load(f)
chunks       = [m['chunk']       for m in metadata_list]
chunk_labels = [m['entity_type'] for m in metadata_list]

try:
    chroma_client = chromadb.EphemeralClient()
except AttributeError:
    chroma_client = chromadb.Client()

existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME, metadata={'hnsw:space': 'cosine'}
)
BATCH = 200
for s in range(0, len(chunks), BATCH):
    e = min(s + BATCH, len(chunks))
    collection.add(
        ids        = [f'chunk_{i:04d}' for i in range(s, e)],
        documents  = chunks[s:e],
        embeddings = ft_embeddings[s:e].tolist(),
        metadatas  = [
            {'entity_type': chunk_labels[i], 'chunk_index': i, 'char_length': len(chunks[i])}
            for i in range(s, e)
        ],
    )
print(f'ChromaDB rebuilt — {collection.count()} docs indexed.')

In [ ]:
# ── Load Groq API key from .env or prompt if needed ────────────────

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Enter Groq API key (hidden): ")
    print("Groq API key received via getpass.")
else:
    print("Groq API key loaded from .env")

if not GROQ_API_KEY.startswith("gsk_"):
    raise ValueError("Invalid key format. Expected 'gsk_...'")
print(f"   Key prefix : {GROQ_API_KEY[:8]}{'*'*20}")

In [ ]:
def retrieve_documents(query: str, n: int = N_RESULTS,
                       filter_type: Optional[str] = None) -> List[Document]:
    q_emb = embedding_model.encode([query], normalize_embeddings=True).tolist()
    where = {'entity_type': filter_type} if filter_type else None
    res   = collection.query(
        query_embeddings=q_emb, n_results=n, where=where,
        include=['documents', 'distances', 'metadatas']
    )
    return [
        Document(
            page_content=doc,
            metadata={**meta, 'similarity': round(1.0 - float(dist), 4)}
        )
        for doc, dist, meta in zip(
            res['documents'][0], res['distances'][0], res['metadatas'][0]
        )
    ]


def format_documents(docs: List[Document]) -> str:
    if not docs:
        return 'No relevant context found.'
    lines = ['=== Supply Chain Knowledge Graph Context ===']
    for i, d in enumerate(docs, 1):
        lines.append(
            f'[Context {i}] {d.metadata.get("entity_type","?")} '
            f'(relevance={d.metadata.get("similarity",0):.3f})'
        )
        lines.append(d.page_content)
    lines.append('=== End of Context ===')
    return '\n'.join(lines)


llm = ChatGroq(
    api_key=GROQ_API_KEY, model_name=GROQ_MODEL_NAME,
    temperature=0.1, max_tokens=1024
)

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', OPTIMISED_SYSTEM),
    ('human', '{context}\n\nQuestion: {question}'),
])

rag_chain = (
    RunnableParallel(
        context  = RunnableLambda(lambda q: format_documents(retrieve_documents(q))),
        question = RunnablePassthrough()
    )
    | rag_prompt | llm | StrOutputParser()
)

print('RAG chain ready with optimised prompt from 03_02_prompt_engineering.')

### Section C — Run 5 Business Queries

In [ ]:
BUSINESS_QUERIES = [
    {
        'id'         : 'BQ-01',
        'question'   : 'Which suppliers have a lead time over 20 days and low reliability? What action should we take?',
        'domain'     : 'Supplier Risk Management',
        'filter_type': None,
    },
    {
        'id'         : 'BQ-02',
        'question'   : 'Which routes are causing delivery delays and what is the business impact?',
        'domain'     : 'Logistics Bottleneck Analysis',
        'filter_type': None,
    },
    {
        'id'         : 'BQ-03',
        'question'   : 'Which warehouses are near full capacity and how should we redistribute inventory?',
        'domain'     : 'Warehouse Capacity Management',
        'filter_type': 'Warehouse',
    },
    {
        'id'         : 'BQ-04',
        'question'   : 'Which products are critically low on stock and need urgent reordering?',
        'domain'     : 'Inventory Replenishment',
        'filter_type': None,
    },
    {
        'id'         : 'BQ-05',
        'question'   : 'What logistics events caused the most significant disruptions and how do we mitigate future risk?',
        'domain'     : 'Supply Chain Risk Mitigation',
        'filter_type': None,
    },
]

In [ ]:
results = []

for bq in BUSINESS_QUERIES:
    print(f'\n{"━"*65}')
    print(f'  {bq["id"]} | {bq["domain"]}')
    print(f'{"━"*65}')
    print(f'  Q: {bq["question"]}\n')

    source_docs = retrieve_documents(
        bq['question'], n=N_RESULTS, filter_type=bq['filter_type']
    )

    t0      = time.time()
    answer  = rag_chain.invoke(bq['question'])
    latency = round(time.time() - t0, 2)

    print(answer)
    print(f'\n  Sources: {len(source_docs)} chunks  |  Latency: {latency}s')

    results.append({
        'id'         : bq['id'],
        'domain'     : bq['domain'],
        'question'   : bq['question'],
        'answer'     : answer,
        'source_docs': [
            {
                'rank'       : i + 1,
                'entity_type': d.metadata.get('entity_type'),
                'similarity' : d.metadata.get('similarity'),
                'chunk'      : d.page_content,
            }
            for i, d in enumerate(source_docs)
        ],
        'latency_s'  : latency,
        'word_count' : len(answer.split()),
    })

print(f'\n{"="*65}')

### Section D — Response Quality Evaluation

In [ ]:
def evaluate_answer(answer: str) -> Dict[str, Any]:
    a = answer.lower()
    groundedness  = min(5, answer.count('Context') + (2 if 'context' in a else 0) + (1 if any(c.isdigit() for c in answer) else 0))
    specificity   = min(5, (2 if any(c.isdigit() for c in answer) else 0) + (2 if any(w in a for w in ['supplier','warehouse','route','product','shipment']) else 0) + (1 if len(answer.split()) > 100 else 0))
    actionability = min(5, (2 if any(w in a for w in ['recommend','action','should','immediately','priorit']) else 0) + (2 if any(w in a for w in ['redistribute','evaluate','contact','review','mitigat']) else 0) + (1 if any(w in a for w in ['because','reason','due to']) else 0))
    structure     = min(5, (2 if 'summary' in a else 0) + (1 if 'finding' in a else 0) + (1 if 'confidence' in a else 0) + (1 if answer.count('•') + answer.count('**') > 2 else 0))
    total = groundedness + specificity + actionability + structure
    return {
        'Groundedness' : groundedness,
        'Specificity'  : specificity,
        'Actionability': actionability,
        'Structure'    : structure,
        'Total'        : total,
        'Pct'          : round(total / 20 * 100, 1),
    }


eval_rows = []
for r in results:
    scores = evaluate_answer(r['answer'])
    eval_rows.append({
        'ID'          : r['id'],
        'Domain'      : r['domain'],
        'Latency (s)' : r['latency_s'],
        'Words'       : r['word_count'],
        'Top-1 Sim'   : r['source_docs'][0]['similarity'] if r['source_docs'] else 0,
        **scores,
    })

df_eval = pd.DataFrame(eval_rows)
print('Response Quality Scores:')
print(df_eval[['ID','Domain','Latency (s)','Words',
               'Groundedness','Specificity','Actionability',
               'Structure','Total','Pct']].to_string(index=False))
print(f'\n   Avg total score : {df_eval["Total"].mean():.1f} / 20')
print(f'   Avg latency     : {df_eval["Latency (s)"].mean():.2f}s')

### Section E — Results Dashboard (PPT-ready Charts)

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    f'Supply Chain RAG Pipeline — Business Query Results\n'
    f'Model: {GROQ_MODEL_NAME} | Prompt: Optimised | Retriever: ChromaDB HNSW',
    fontsize=13, fontweight='bold'
)
gs     = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
colors = ['#4A90D9','#7ED321','#F5A623','#9B59B6','#E74C3C']
ids    = df_eval['ID'].tolist()
x      = np.arange(len(ids))

ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(ids, df_eval['Total'], color=colors, edgecolor='white')
for bar, val, pct in zip(bars, df_eval['Total'], df_eval['Pct']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val}/20\n({pct}%)', ha='center', fontsize=8)
ax1.set_ylim(0, 22)
ax1.set_title('Quality Score per Query')
ax1.set_ylabel('Total Score (max 20)')
ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(ids, df_eval['Latency (s)'], color=colors, edgecolor='white')
for i, v in enumerate(df_eval['Latency (s)']):
    ax2.text(i, v + 0.05, f'{v}s', ha='center', fontsize=8)
ax2.set_title('API Latency per Query')
ax2.set_ylabel('Seconds')
ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(ids, df_eval['Top-1 Sim'], color=colors, edgecolor='white')
ax3.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='0.5 threshold')
ax3.set_ylim(0, 1)
ax3.set_title('Top-1 Retrieval Similarity')
ax3.set_ylabel('Cosine Similarity')
ax3.legend(fontsize=8)
ax3.grid(axis='y', alpha=0.3)

ax4 = fig.add_subplot(gs[1, :2])
dims      = ['Groundedness','Specificity','Actionability','Structure']
dim_cols  = ['#4A90D9','#7ED321','#F5A623','#9B59B6']
w         = 0.18
for j, (dim, dc) in enumerate(zip(dims, dim_cols)):
    ax4.bar(x + j*w - 1.5*w, df_eval[dim], w,
            label=dim, color=dc, edgecolor='white', alpha=0.85)
ax4.set_xticks(x)
ax4.set_xticklabels(ids)
ax4.set_ylim(0, 5)
ax4.set_ylabel('Score (0-5)')
ax4.set_title('Quality Breakdown by Dimension')
ax4.legend(fontsize=8)
ax4.grid(axis='y', alpha=0.3)

ax5 = fig.add_subplot(gs[1, 2])
ax5.scatter(df_eval['Words'], df_eval['Total'],
            c=colors, s=150, zorder=5, edgecolors='white', linewidths=1.2)
for _, row in df_eval.iterrows():
    ax5.annotate(row['ID'], (row['Words'], row['Total']),
                 textcoords='offset points', xytext=(5, 3), fontsize=8)
ax5.set_xlabel('Answer Word Count')
ax5.set_ylabel('Total Quality Score')
ax5.set_title('Length vs Quality')
ax5.grid(alpha=0.3)

plt.savefig(OUTPUT_DIR / '03_03_results_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved → outputs/03_03_results_dashboard.png')

In [ ]:
entity_types = ['Supplier','Product','Warehouse','Route','Order',
                'Shipment','Inventory','Customer','LogisticsEvent']

heatmap_data = np.zeros((len(results), len(entity_types)))
for i, r in enumerate(results):
    for doc in r['source_docs']:
        et = doc.get('entity_type', 'Other')
        if et in entity_types:
            heatmap_data[i, entity_types.index(et)] += 1

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(heatmap_data, cmap='Blues', aspect='auto', vmin=0, vmax=N_RESULTS)
ax.set_xticks(range(len(entity_types)))
ax.set_xticklabels(entity_types, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(results)))
ax.set_yticklabels([r['id'] for r in results])
plt.colorbar(im, ax=ax, label='Chunks retrieved')
for i in range(len(results)):
    for j in range(len(entity_types)):
        if heatmap_data[i, j] > 0:
            ax.text(j, i, int(heatmap_data[i, j]),
                    ha='center', va='center', fontsize=9, color='white')
ax.set_title('Retrieved Entity Types per Business Query', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_03_retrieval_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved → outputs/03_03_retrieval_heatmap.png')

### Section F — Export Report & Artefact Checklist

In [ ]:
RESULTS_FILE = OUTPUT_DIR / '03_03_rag_results.json'
export = {
    'run_metadata': {
        'groq_model'      : GROQ_MODEL_NAME,
        'prompt_strategy' : pe['best_strategy'],
        'retriever'       : 'ChromaDB HNSW (cosine)',
        'embedding_model' : FINE_TUNED_MODEL,
        'n_results'       : N_RESULTS,
        'corpus_size'     : len(chunks),
    },
    'query_results': results,
    'evaluation'   : eval_rows,
}
with open(RESULTS_FILE, 'w') as f:
    json.dump(export, f, indent=2)
print(f'Full results saved → {RESULTS_FILE}')

In [ ]:
REPORT_FILE = OUTPUT_DIR / '03_03_rag_answers_report.txt'
lines = [
    'SUPPLY CHAIN RAG PIPELINE — BUSINESS QUERY ANSWERS',
    f'Model: {GROQ_MODEL_NAME} | Strategy: {pe["best_strategy"]}',
    '=' * 70,
]
for r in results:
    lines += [
        f'\n{r["id"]} — {r["domain"]}',
        f'Q: {r["question"]}',
        '-' * 70,
        r['answer'],
        f'\nSources ({len(r["source_docs"])}):', 
    ]
    for d in r['source_docs']:
        lines.append(
            f'  [{d["rank"]}] {d["entity_type"]} (sim={d["similarity"]}) '
            f'— {d["chunk"][:70]}...'
        )
    lines.append('=' * 70)

with open(REPORT_FILE, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'Text report saved → {REPORT_FILE}')

In [ ]:
print('=' * 65)
print('  RAG Business Queries & Evaluation')
print('=' * 65)
print(f'\n  Queries answered   : {len(results)}')
print(f'  Avg quality score  : {df_eval["Total"].mean():.1f} / 20  ({df_eval["Pct"].mean():.0f}%)')
print(f'  Avg latency        : {df_eval["Latency (s)"].mean():.2f}s')
print(f'  Avg top-1 sim      : {df_eval["Top-1 Sim"].mean():.4f}')
print(f'\n  Artefacts saved:')
artefacts = [
    'outputs/03_03_results_dashboard.png   — PPT-ready quality dashboard',
    'outputs/03_03_retrieval_heatmap.png   — entity retrieval heatmap',
    'outputs/03_03_rag_results.json        — full structured results',
    'outputs/03_03_rag_answers_report.txt  — slide notes text file',
]
for a in artefacts:
    print(f'   {a}')
print('=' * 65)